# 04 — Grad-CAM Explainability

Theory: [`docs/06_gradcam.md`](../docs/06_gradcam.md)

In [ ]:
# --- environment setup: works on Kaggle, Colab and locally ---
import sys, os, glob
REPO = "https://github.com/Mohd2Shaan/Deepfake-Detection-Project.git"
if os.path.exists("/kaggle/input"):                      # ---- Kaggle
    if not os.path.exists("/kaggle/working/project"):
        !git clone -q {REPO} /kaggle/working/project
    %cd /kaggle/working/project
    !pip install -q grad-cam
    # use the dataset attached via "Add Input" directly (read-only, nothing to download)
    for depth in range(1, 5):
        hits = [os.path.dirname(t) for t in glob.glob("/kaggle/input/" + "*/" * depth + "train")
                if os.path.isdir(os.path.join(os.path.dirname(t), "valid"))]
        if hits:
            os.environ["DEEPFAKE_DATA_DIR"] = hits[0]
            break
    else:
        print("WARNING: dataset not found - attach '140k Real and Fake Faces' via Add Input")
elif "google.colab" in sys.modules:                       # ---- Colab
    if not os.path.exists("/content/project"):
        !git clone -q {REPO} /content/project
    %cd /content/project
    !pip install -q -r requirements.txt kagglehub
elif os.path.basename(os.getcwd()) == "notebooks":        # ---- local
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
# --- dataset (Colab/local: downloads ~4 GB via kagglehub the first time; Kaggle: already attached) ---
from src import config
if not (config.DATA_DIR / "train").exists():
    !python scripts/download_dataset.py --move
print("data dir:", config.DATA_DIR)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from PIL import Image
from src import config
from src.model import load_checkpoint, get_target_layer
from src.gradcam import GradCAMExplainer, manual_gradcam, save_heatmap_grid, pick_correct_examples

device = config.get_device()
model, ckpt = load_checkpoint(config.CHECKPOINT_DIR / 'efficientnet_best.pth', device)
name = ckpt['model_name']
explainer = GradCAMExplainer(model, name, device)

## 1. Correctly classified real and fake faces

In [ ]:
!python -m src.gradcam --checkpoint checkpoints/efficientnet_best.pth --n 12
from IPython.display import Image as Show, display
for p in sorted(config.HEATMAP_DIR.glob(f'{name}_*.png')):
    display(Show(str(p)))

## 2. Library Grad-CAM vs hand-written Grad-CAM
Both should produce (almost) the same map — shows we understand what the library computes.

In [ ]:
import numpy as np
path = pick_correct_examples(explainer, config.DATA_DIR, 1, 'fake')[0]
img = Image.open(path)
lib = explainer.explain(img, target_class=1)['cam']
x = explainer.transform(img.convert('RGB')).unsqueeze(0).to(device)
mine = manual_gradcam(model, get_target_layer(model, name), x, sign=1.0)
print('correlation:', np.corrcoef(lib.ravel(), mine.ravel())[0, 1])
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
ax[0].imshow(img); ax[1].imshow(lib, cmap='jet'); ax[2].imshow(mine, cmap='jet')
for a, t in zip(ax, ['image', 'pytorch-grad-cam', 'manual']): a.set_title(t); a.axis('off')

## 3. ResNet-18 heatmaps for comparison

In [ ]:
!python -m src.gradcam --checkpoint checkpoints/resnet_best.pth --n 8

## Observations

_Where does the model look for fakes? (eyes, hairline, teeth, background edges?) Where does it look on misclassified images?_